# Comparativo de Modelos de Embedding PT-BR

> **Objetivo**: Avaliar modelos de embedding para notícias governamentais brasileiras
>
> **Issue**: [#1 - Comparativo de Modelos de Embedding PT-BR](https://github.com/destaquesgovbr/data-science/issues/1)
>
> **Responsável**: Luis Felipe de Moraes

---

## 📋 Estrutura do Notebook

1. [Setup e Configuração](#1-setup-e-configuração)
2. [Carregamento de Modelos](#2-carregamento-de-modelos)
3. [Teste Rápido com Exemplos](#3-teste-rápido-com-exemplos)
4. [Preparação do Dataset](#4-preparação-do-dataset)
5. [Geração de Embeddings](#5-geração-de-embeddings)
6. [Avaliação Quantitativa](#6-avaliação-quantitativa)
7. [Avaliação Qualitativa](#7-avaliação-qualitativa)
8. [Benchmark de Performance](#8-benchmark-de-performance)
9. [Visualizações](#9-visualizações)
10. [Conclusões e Recomendações](#10-conclusões-e-recomendações)

---

## 1. Setup e Configuração

Imports necessários e configuração do ambiente.

In [ ]:
# Imports básicos
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Sentence Transformers
from sentence_transformers import SentenceTransformer, util

# Métricas
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr, spearmanr

# Scripts locais
sys.path.append(str(Path.cwd().parent / 'scripts'))
from load_models import MODELS_TO_EVALUATE, load_model, load_all_models

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Criar diretórios de resultados se não existirem
RESULTS_DIR = Path.cwd().parent / 'results'
EMBEDDINGS_DIR = RESULTS_DIR / 'embeddings'
METRICS_DIR = RESULTS_DIR / 'metrics'
VIZ_DIR = RESULTS_DIR / 'visualizations'

for dir_path in [EMBEDDINGS_DIR, METRICS_DIR, VIZ_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✓ {dir_path}")

## 2. Carregamento de Modelos

Carregar todos os modelos de embedding para comparação.

In [ ]:
# Listar modelos disponíveis
print("\n" + "="*100)
print("MODELOS DISPONÍVEIS PARA AVALIAÇÃO")
print("="*100 + "\n")

df_models = pd.DataFrame([
    {
        'Nome': m.name,
        'Model ID': m.model_id,
        'Categoria': m.category,
        'Dimensões': m.dimensions,
        'Max Tokens': m.max_tokens,
        'Parâmetros (M)': m.parameters_millions,
    }
    for m in MODELS_TO_EVALUATE
])

display(df_models)

In [ ]:
# Escolher quais modelos carregar
# Opções: 'multilingual', 'pt-specific', ou None para todos
CATEGORIES_TO_LOAD = None  # None = todos

# Se quiser testar apenas alguns modelos primeiro, descomente:
# CATEGORIES_TO_LOAD = ['pt-specific']  # Apenas PT-BR
# CATEGORIES_TO_LOAD = ['multilingual']  # Apenas multilinguais

In [ ]:
# Carregar modelos (pode demorar alguns minutos)
print("\n⏳ Carregando modelos... (isso pode demorar)\n")

loaded_models = load_all_models(
    device=DEVICE,
    categories=CATEGORIES_TO_LOAD
)

print(f"\n✓ {len(loaded_models)} modelos carregados com sucesso!")
print(f"\nModelos disponíveis: {list(loaded_models.keys())}")

## 3. Teste Rápido com Exemplos

Testar modelos com sentenças exemplo de domínio governamental.

In [ ]:
# Sentenças de teste (contexto governamental brasileiro)
test_sentences = [
    # Par 1: Muito similares (mesmo tópico, sinônimos)
    "Ministério da Educação anuncia novo programa de bolsas para estudantes",
    "MEC apresenta iniciativa de financiamento educacional para alunos",
    
    # Par 2: Relacionados (mesmo domínio, contextos diferentes)
    "Governo federal implementa medidas de combate à pandemia de COVID-19",
    "Ministério da Saúde distribui vacinas para estados e municípios",
    
    # Par 3: Diferentes (domínios completamente distintos)
    "Banco Central eleva taxa Selic para controlar inflação",
    "Ministério do Meio Ambiente cria novas unidades de conservação na Amazônia",
    
    # Sentenças adicionais para teste
    "Portaria normativa estabelece diretrizes para concessão de benefícios",
    "Presidente da República sanciona projeto de lei sobre reforma tributária",
    "INSS anuncia calendário de pagamentos para aposentados e pensionistas",
    "Polícia Federal deflagra operação contra corrupção em órgãos públicos",
]

print("\n📋 SENTENÇAS DE TESTE:\n")
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. {sent}")

In [ ]:
# Gerar embeddings de teste para todos os modelos
print("\n🧪 Gerando embeddings de teste...\n")

test_embeddings = {}

for model_name, (config, model) in loaded_models.items():
    print(f"  {model_name}...", end=' ')
    embeddings = model.encode(
        test_sentences,
        convert_to_tensor=True,
        show_progress_bar=False
    )
    test_embeddings[model_name] = embeddings
    print(f"✓ Shape: {embeddings.shape}")

print("\n✓ Embeddings gerados!")

In [ ]:
# Calcular matriz de similaridade para cada modelo
print("\n📊 MATRIZES DE SIMILARIDADE (Cosine)\n")

# Pegar apenas 6 primeiras sentenças para visualização
n_sentences = 6

for model_name, embeddings in list(test_embeddings.items())[:3]:  # Mostrar apenas 3 modelos
    print(f"\n{'='*80}")
    print(f"Modelo: {model_name}")
    print(f"{'='*80}\n")
    
    # Calcular similaridade
    emb_subset = embeddings[:n_sentences]
    sim_matrix = util.cos_sim(emb_subset, emb_subset).cpu().numpy()
    
    # Criar heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(
        sim_matrix,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        square=True,
        cbar_kws={'label': 'Cosine Similarity'},
        ax=ax
    )
    
    # Labels
    labels = [f"S{i+1}" for i in range(n_sentences)]
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_title(f'Matriz de Similaridade - {model_name}')
    
    plt.tight_layout()
    plt.savefig(VIZ_DIR / f'similarity_matrix_{model_name.lower().replace("-", "_")}.png', dpi=150)
    plt.show()
    
    # Mostrar pares mais similares
    print("\nPares mais similares (excluindo diagonal):")
    for i in range(n_sentences):
        for j in range(i+1, n_sentences):
            print(f"  S{i+1} ↔ S{j+1}: {sim_matrix[i, j]:.3f}")

### 💡 Análise Preliminar

**Esperado:**
- S1 ↔ S2: Alta similaridade (MEC/educação, sinônimos)
- S3 ↔ S4: Similaridade moderada (saúde/pandemia, relacionados)
- S5 ↔ S6: Baixa similaridade (economia vs meio ambiente)

**Observações:**
- Modelos que capturam melhor essas nuances têm melhor compreensão semântica
- Atenção especial a jargão governamental (MEC, Selic, INSS, etc)

## 4. Preparação do Dataset

Carregar dataset de notícias governamentais para avaliação.

**TODO**: 
- [ ] Definir fonte de dados (database, CSV, parquet)
- [ ] Criar queries de teste representativas
- [ ] Anotar relevância (ground truth)

In [ ]:
# Placeholder - Carregar dataset real
# TODO: Implementar carregamento de notícias do data-platform

# Exemplo de estrutura esperada:
# df_noticias = pd.read_parquet('path/to/noticias.parquet')
# Colunas esperadas: ['id', 'titulo', 'lead', 'corpo', 'orgao', 'data', 'url']

print("⚠️  TODO: Implementar carregamento do dataset de notícias")
print("\nPor enquanto, use sentenças de teste da seção anterior.")

In [ ]:
# Criar queries de teste
# TODO: Definir queries representativas do domínio

test_queries = [
    # Jargão governamental
    "portaria normativa medida provisória",
    "decreto presidencial legislação federal",
    
    # Siglas
    "SUS UBS ACS programa saúde",
    "MEC educação ensino fundamental",
    "INSS aposentadoria benefícios previdenciários",
    
    # Tópicos específicos
    "vacinação covid-19 imunização",
    "reforma tributária impostos arrecadação",
    "segurança pública polícia federal criminalidade",
    "meio ambiente desmatamento Amazônia",
    "agricultura familiar crédito rural",
]

print(f"\n📋 {len(test_queries)} queries de teste definidas")

## 5. Geração de Embeddings

Gerar embeddings para todo o corpus e queries.

In [ ]:
# Gerar embeddings das queries para todos os modelos
print("\n🔄 Gerando embeddings das queries...\n")

query_embeddings = {}

for model_name, (config, model) in loaded_models.items():
    print(f"  {model_name}...", end=' ')
    embeddings = model.encode(
        test_queries,
        convert_to_tensor=True,
        show_progress_bar=False,
        batch_size=32
    )
    query_embeddings[model_name] = embeddings
    print(f"✓ {embeddings.shape}")

print("\n✓ Query embeddings gerados!")

In [ ]:
# TODO: Gerar embeddings do corpus completo
# ATENÇÃO: Isso pode demorar horas dependendo do tamanho do corpus!

# corpus_embeddings = {}
# for model_name, (config, model) in loaded_models.items():
#     print(f"\nGerando embeddings para {model_name}...")
#     embeddings = model.encode(
#         df_noticias['texto'].tolist(),
#         show_progress_bar=True,
#         batch_size=32,
#         convert_to_tensor=True
#     )
#     corpus_embeddings[model_name] = embeddings
#     
#     # Salvar embeddings
#     np.save(EMBEDDINGS_DIR / f'{model_name}_corpus.npy', embeddings.cpu().numpy())

print("⚠️  TODO: Implementar geração de embeddings do corpus completo")

## 6. Avaliação Quantitativa

Calcular métricas de retrieval: NDCG@10, MAP, MRR, Recall@K

In [ ]:
# Funções de métrica

def calculate_ndcg_at_k(relevances, k=10):
    """
    Calcula NDCG@k (Normalized Discounted Cumulative Gain)
    
    Args:
        relevances: Lista de scores de relevância dos documentos retornados
        k: Número de documentos top a considerar
    """
    relevances = np.array(relevances)[:k]
    
    # DCG
    dcg = relevances[0] + np.sum(relevances[1:] / np.log2(np.arange(2, len(relevances) + 1)))
    
    # IDCG (ideal)
    ideal_relevances = np.sort(relevances)[::-1]
    idcg = ideal_relevances[0] + np.sum(ideal_relevances[1:] / np.log2(np.arange(2, len(ideal_relevances) + 1)))
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


def calculate_map(relevances_list):
    """
    Calcula MAP (Mean Average Precision)
    
    Args:
        relevances_list: Lista de listas de relevâncias binárias (0/1)
    """
    average_precisions = []
    
    for relevances in relevances_list:
        relevances = np.array(relevances)
        if relevances.sum() == 0:
            continue
            
        precisions = []
        num_relevant = 0
        
        for i, rel in enumerate(relevances, 1):
            if rel == 1:
                num_relevant += 1
                precisions.append(num_relevant / i)
        
        if precisions:
            average_precisions.append(np.mean(precisions))
    
    return np.mean(average_precisions) if average_precisions else 0.0


def calculate_mrr(relevances_list):
    """
    Calcula MRR (Mean Reciprocal Rank)
    
    Args:
        relevances_list: Lista de listas de relevâncias binárias (0/1)
    """
    reciprocal_ranks = []
    
    for relevances in relevances_list:
        for i, rel in enumerate(relevances, 1):
            if rel == 1:
                reciprocal_ranks.append(1.0 / i)
                break
    
    return np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0


def calculate_recall_at_k(relevances_list, k=10):
    """
    Calcula Recall@k
    
    Args:
        relevances_list: Lista de listas de relevâncias binárias (0/1)
        k: Número de documentos top a considerar
    """
    recalls = []
    
    for relevances in relevances_list:
        relevances = np.array(relevances)
        total_relevant = relevances.sum()
        
        if total_relevant == 0:
            continue
            
        retrieved_relevant = relevances[:k].sum()
        recalls.append(retrieved_relevant / total_relevant)
    
    return np.mean(recalls) if recalls else 0.0

print("✓ Funções de métrica definidas")

In [ ]:
# TODO: Implementar avaliação com ground truth

# Exemplo de estrutura:
# results = []
# 
# for model_name in loaded_models.keys():
#     # Para cada query
#     for query_idx, query in enumerate(test_queries):
#         # Buscar top-K documentos
#         similarities = util.cos_sim(
#             query_embeddings[model_name][query_idx],
#             corpus_embeddings[model_name]
#         )[0]
#         
#         top_k = torch.topk(similarities, k=20)
#         doc_ids = top_k.indices.cpu().tolist()
#         
#         # Comparar com ground truth
#         relevances = get_relevances(query_idx, doc_ids)  # Função para pegar relevância anotada
#         
#         # Calcular métricas
#         ndcg = calculate_ndcg_at_k(relevances, k=10)
#         
#         results.append({
#             'model': model_name,
#             'query_idx': query_idx,
#             'ndcg@10': ndcg,
#         })

print("⚠️  TODO: Implementar avaliação quantitativa com ground truth")

## 7. Avaliação Qualitativa

Análise de casos específicos e comportamento dos modelos.

In [ ]:
# Analisar queries específicas
query_to_analyze = "SUS UBS ACS programa saúde"

print(f"\n🔍 ANÁLISE QUALITATIVA\n")
print(f"Query: \"{query_to_analyze}\"\n")
print("="*80)

query_idx = test_queries.index(query_to_analyze)

# Para cada modelo, mostrar top-5 resultados mais similares
for model_name in list(loaded_models.keys())[:3]:  # Mostrar apenas 3 modelos
    print(f"\nModelo: {model_name}")
    print("-" * 80)
    
    # Calcular similaridade com sentenças de teste
    query_emb = query_embeddings[model_name][query_idx]
    test_emb = test_embeddings[model_name]
    
    similarities = util.cos_sim(query_emb, test_emb)[0]
    top_results = torch.topk(similarities, k=5)
    
    print("\nTop-5 sentenças mais similares:\n")
    for rank, (score, idx) in enumerate(zip(top_results.values, top_results.indices), 1):
        print(f"{rank}. [{score:.3f}] {test_sentences[idx][:80]}...")

print("\n" + "="*80)

### 💡 Análise de Casos Específicos

**Observar:**
1. **Jargão**: Modelos entendem siglas (SUS, MEC, INSS)?
2. **Sinônimos**: Capturam "educação" ↔ "ensino"?
3. **Contexto BR**: Diferenciam PT-BR de PT-PT?
4. **Estrutura**: Reconhecem "portaria", "decreto", "medida provisória"?

## 8. Benchmark de Performance

Medir velocidade de encoding e uso de recursos.

In [ ]:
import time

# Benchmark de velocidade
print("\n⏱️  BENCHMARK DE PERFORMANCE\n")
print("="*80 + "\n")

benchmark_texts = test_sentences * 10  # 100 sentenças
batch_sizes = [1, 16, 32, 64]

benchmark_results = []

for model_name, (config, model) in loaded_models.items():
    print(f"\nModelo: {model_name}")
    print("-" * 80)
    
    for batch_size in batch_sizes:
        # Warmup
        model.encode(benchmark_texts[:batch_size], show_progress_bar=False)
        
        # Benchmark
        start = time.time()
        embeddings = model.encode(
            benchmark_texts,
            batch_size=batch_size,
            show_progress_bar=False
        )
        elapsed = time.time() - start
        
        throughput = len(benchmark_texts) / elapsed
        
        benchmark_results.append({
            'model': model_name,
            'batch_size': batch_size,
            'time_seconds': elapsed,
            'throughput_docs_per_sec': throughput,
        })
        
        print(f"  Batch {batch_size:>2}: {elapsed:.2f}s ({throughput:.1f} docs/s)")

# Criar DataFrame
df_benchmark = pd.DataFrame(benchmark_results)
df_benchmark.to_csv(METRICS_DIR / 'benchmark_performance.csv', index=False)

print("\n✓ Benchmark completo!")

In [ ]:
# Visualizar benchmark
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Throughput por modelo (batch_size=32)
df_best_batch = df_benchmark[df_benchmark['batch_size'] == 32].sort_values('throughput_docs_per_sec', ascending=False)

axes[0].barh(df_best_batch['model'], df_best_batch['throughput_docs_per_sec'])
axes[0].set_xlabel('Throughput (docs/segundo)')
axes[0].set_title('Performance por Modelo (batch_size=32)')
axes[0].grid(axis='x', alpha=0.3)

# Throughput vs batch size
for model_name in df_benchmark['model'].unique():
    df_model = df_benchmark[df_benchmark['model'] == model_name]
    axes[1].plot(df_model['batch_size'], df_model['throughput_docs_per_sec'], marker='o', label=model_name)

axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Throughput (docs/segundo)')
axes[1].set_title('Throughput vs Batch Size')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_DIR / 'benchmark_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Visualizações

Criar visualizações comparativas dos modelos.

In [ ]:
# TODO: Criar visualizações após avaliação completa

# Exemplos de visualizações a criar:
# 1. Barplot: NDCG@10 por modelo
# 2. Heatmap: Performance por categoria de query
# 3. Scatter: Dimensões vs Performance
# 4. Boxplot: Distribuição de scores de similaridade
# 5. Radar chart: Trade-offs (qualidade, velocidade, dimensões)

print("⚠️  TODO: Criar visualizações após avaliação completa")

## 10. Conclusões e Recomendações

Síntese dos resultados e recomendações por caso de uso.

In [ ]:
# TODO: Compilar resultados finais

print("\n" + "="*80)
print("SÍNTESE DE RESULTADOS")
print("="*80 + "\n")

print("⚠️  TODO: Compilar análise final após todas as avaliações")

print("""
Estrutura esperada:

1. RANKING GERAL
   - Por NDCG@10
   - Por MAP
   - Por velocidade

2. TRADE-OFFS IDENTIFICADOS
   - Qualidade vs Velocidade
   - Dimensões vs Performance
   - Multilingual vs PT-específico

3. RECOMENDAÇÕES POR CASO DE USO
   - Busca geral: [Modelo X]
   - Documentos longos: [Modelo Y]
   - Baixa latência: [Modelo Z]
   - RAG: [Modelo W]

4. PRÓXIMOS PASSOS
   - Fine-tuning do modelo escolhido?
   - Teste com corpus maior?
   - Avaliação com usuários reais?
""")

---

## 📝 Notas e Observações

Use esta seção para anotar insights durante a experimentação:

### Observações Gerais
- 

### Modelos Multilinguais
- BGE-M3:
- E5-Large:
- GTE-Multilingual:

### Modelos PT-BR
- Serafim-900M:
- BERTimbau:
- Jina v2 PT:

### Surpresas/Descobertas
- 

---

**Notebook criado em**: 2026-03-05  
**Última atualização**: 2026-03-05  
**Versão**: 1.0